# recs_025 -- USE-controlled ablation: blend-weight x pooling variant (embedder-fair vs. two_tower_v1)

## Executive Summary

- `recs_024`'s bge-small win is confounded: `two_tower_v1` fine-tunes USE end-to-end and was
  never retrained on bge-small, so we don't know if it's RAG's architecture winning or just a
  free embedder upgrade.
- This notebook holds the embedder fixed at USE and sweeps `description_blend_weight`
  (placeholder 0.1, never tuned) and the 3 untested pooling variants, to see if pipeline tuning
  alone can close the gap.
- **Result: nothing in the 24-cell grid beats `two_tower_v1`.** Best cell (`any_polarity__flat`,
  blend=0.2-0.3) is a marginal +0.001-0.004 nudge over the untuned default -- still ~0.04 short
  of `two_tower_v1` on Slice B Hit@K. This supports (without proving) that `recs_024`'s win was
  a genuine embedder-quality effect, not pipeline-tuning luck.

## Business Context

`recs_024` answers "does this beat the currently-shipped `two_tower_v1`," not "is RAG's approach
better." That needs the same embedder on both sides -- but retraining `two_tower_v1` on
bge-small is a real training job, not a re-score. Cheaper alternative: see how far USE-based RAG
can get on its own, since `two_tower_v1` is already a USE-based bar.

## Research Question

Holding the embedder fixed at USE (matching `two_tower_v1`), does tuning `description_blend_weight`
and/or switching pooling variant close some or all of the gap between RAG retrieval and
`two_tower_v1` on the primary Slice A/B metrics?

## Hypothesis

Two untested choices are being tested together:

1. **`description_blend_weight`** -- a placeholder (0.1, never tuned). No directional guess.
2. **Pooling variant** -- `any_polarity` vs `recommended_only`; `flat` vs `log_weighted`. Only
   `any_polarity__flat` had ever been evaluated. Guessed `log_weighted` helps (less noisy than
   flat), `recommended_only` is a toss-up.

Combined prediction: something in the 24-cell grid beats `rag_v1`'s baseline
(0.475/0.453/0.456), but unclear if anything closes the gap to `two_tower_v1`
(0.512/0.460/0.496).

**Result:**
1. Blend weight -- confirmed, barely (0.2-0.3 edges out 0.1).
2. Pooling variant -- wrong: both alternatives underperformed the simple baseline.
3. Gap-closing -- no: nothing gets close to `two_tower_v1`.

## Definitions

| Term | Meaning |
|---|---|
| `description_blend_weight` | `(1-w)*pooled_reviews + w*description`, then L2-normalized. Controls how much a game's IGDB description (vs. its pooled review text) drives its profile vector. |
| `any_polarity` / `recommended_only` | Whether negative reviews are included in the pooled review set (`any_polarity`) or filtered out (`recommended_only`, Ablation A2). |
| `flat` / `log_weighted` | Uniform mean pooling vs. `log1p(votes_helpful)`-weighted pooling (Ablation A). |
| `Hit@100` / `Recall@100` | Retrieval-family metrics (`eval_retrieval_*`), capped at `k_retrieval=100` -- these RAG methods aren't reranked, so this is the metric family they're actually compared on. |
| Slice A / Slice B | `slice_a_multi_target` (primary metric Recall@K) vs. `slice_b_single_target` (primary metric Hit@K), per `docs/recommendation_evaluation_overview.md`. |

## Data Sources

| Source | Role |
|---|---|
| `artifacts/recs/embeddings/game_chunks/default/game_review_chunks.parquet` | Stage 1 chunk table (review + description text) -- embedder-agnostic, reused as-is. |
| `artifacts/recs/offline_eval/runs/rag_v1/eval_offline_examples.jsonl` | The same 12,500-example val cohort used throughout Stage 3 -- reused so every notebook in this series is comparable on identical examples. |
| `data/processed/steam_reviews_cleaned_english_val_norm.parquet` | Val split -- rejoined on `(user_id, query_app_id)` to recover each example's raw query review text. |
| `artifacts/recs/offline_eval/runs/rag_v1/eval_retrieval_overall.csv`, `eval_retrieval_by_slice.csv` | `two_tower_v1`'s bar numbers, pulled directly (unaffected by this notebook, still USE-based/unchanged in production). |

## Design / Process

1. Load Stage 1 chunk table + eval cohort, recover `query_plus_desc` text (same as `recs_024`).
2. Embed all chunks + all queries with USE once -- query embedding doesn't depend on
   pooling/blend, so this is the only expensive step, shared across every grid cell.
   L2-normalize each row right after encoding (matches Stage 2 exactly, unlike `recs_024`'s
   simplified normalize-after-pool-only).
3. For each of 4 pooling variants x 6 blend weights: pool, blend, L2-normalize -- mirrors
   `recs_job_game_chunk_embeddings.py`, reimplemented inline.
4. Score every cell against the fixed query embeddings (cosine sim, self-masked, top 100).
5. Sanity check: `any_polarity__flat`/blend=0.1 should reproduce `rag_v1`'s real numbers.
6. Compare the full grid against `two_tower_v1`.

## Evaluation Outputs / Artifacts

| Artifact | Description |
|---|---|
| Comparison grid (this notebook, Analysis section) | Hit@100 / Recall@100, overall and by slice, for all 4 pooling variants x 6 blend weights, plus the `two_tower_v1` and current-baseline reference rows. |

## Notebook Roadmap

1. Setup
2. Load chunk table + eval cohort, recover query text
3. Embed catalog + queries with USE (one pass, shared across all grid cells)
4. Pool + blend + score the full grid
5. Comparison table + findings

# Analysis

## Setup

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

def _find_repo_root(start: Path) -> Path:
    p = start.resolve()
    while not (p / "pyproject.toml").is_file():
        if p.parent == p:
            raise RuntimeError("Could not find repo root (pyproject.toml not found).")
        p = p.parent
    return p

REPO_ROOT = _find_repo_root(Path.cwd())
import sys
sys.path.insert(0, str(REPO_ROOT / "src"))

from steam_review_ml.recommender.math_utils import l2_normalize
from steam_review_ml.evaluation.retrieval_offline_eval import hit_rate_at_k, precision_at_k, recall_at_k

RUN_DIR = REPO_ROOT / "artifacts" / "recs" / "offline_eval" / "runs" / "rag_v1"
CHUNKS_PATH = REPO_ROOT / "artifacts" / "recs" / "embeddings" / "game_chunks" / "default" / "game_review_chunks.parquet"
VAL_SPLIT_PATH = REPO_ROOT / "data" / "processed" / "steam_reviews_cleaned_english_val_norm.parquet"

USER_COL = "author.steamid"
K_RETRIEVAL = 100
BLEND_WEIGHTS = [0.0, 0.05, 0.1, 0.2, 0.3, 0.5]
POLARITY_ARMS = ("any_polarity", "recommended_only")
WEIGHTING_ARMS = ("flat", "log_weighted")

pd.options.display.max_colwidth = 60
print(f"REPO_ROOT={REPO_ROOT}")

REPO_ROOT=/home/ryanr/workspace/steam_recommendations


## Load Stage 1 Chunk Table + Eval Cohort, Recover Query Text

In [2]:
chunks_df = pd.read_parquet(CHUNKS_PATH)
review_chunks = chunks_df[chunks_df["chunk_type"] == "review"].reset_index(drop=True)
description_by_app: dict[int, str] = dict(
    zip(
        chunks_df.loc[chunks_df["chunk_type"] == "description", "app_id"],
        chunks_df.loc[chunks_df["chunk_type"] == "description", "text"],
    )
)
print(f"chunk rows: {len(chunks_df):,} (review={len(review_chunks):,}, description={len(description_by_app):,})")

examples = []
with open(RUN_DIR / "eval_offline_examples.jsonl") as f:
    for line in f:
        rec = json.loads(line)
        if rec["method"] != "rag_chunk_v1_query_plus_desc":
            continue
        examples.append(
            {
                "ex_idx": rec["ex_idx"],
                "user_id": rec["user_id"],
                "query_app_id": rec["query_app_id"],
                "positives": set(json.loads(rec["validation_positive_app_ids_json"])),
                "n_eval_targets": rec["n_eval_targets"],
                "slice_name": rec["slice_name"],
            }
        )
examples_df = pd.DataFrame(examples)
print(f"eval cohort rows: {len(examples_df):,}")

val_df = pd.read_parquet(VAL_SPLIT_PATH, columns=[USER_COL, "app_id", "review"])
val_df["_key"] = val_df[USER_COL].astype(str) + "::" + val_df["app_id"].astype(str)
review_text_by_key = dict(zip(val_df["_key"], val_df["review"]))

def query_plus_desc_text(user_id: str, query_app_id: int) -> str:
    key = f"{user_id}::{query_app_id}"
    query_text = review_text_by_key.get(key)
    if query_text is None:
        raise KeyError(f"No val-split review found for {key!r}")
    description = description_by_app.get(int(query_app_id))
    if description:
        return f"{query_text}\n\n{description}"
    return query_text

examples_df["query_text"] = examples_df.apply(
    lambda r: query_plus_desc_text(r["user_id"], r["query_app_id"]), axis=1
)
display(examples_df[["ex_idx", "query_app_id", "n_eval_targets", "slice_name"]].head(3))

chunk rows: 16,010 (review=15,695, description=315)
eval cohort rows: 12,500


   ex_idx  query_app_id  n_eval_targets             slice_name
0       0        812140               1  slice_b_single_target
1       1        485510               1  slice_b_single_target
2       2        646570               2   slice_a_multi_target

## Embed Catalog + Queries With USE (One Pass, Shared Across All Grid Cells)

In [3]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
import tensorflow as tf
import tensorflow_hub as hub

for gpu in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception:
        pass

USE_URL = "https://tfhub.dev/google/universal-sentence-encoder/4"
use_embed_fn = hub.load(USE_URL)


def encode_use(texts: list[str], *, batch_size: int = 64, max_chars: int = 8000) -> np.ndarray:
    """Batch-encode and L2-normalize each row -- matches Stage 2's _encode_texts exactly."""
    clipped = [t[:max_chars] for t in texts]
    chunks = []
    for start in range(0, len(clipped), batch_size):
        batch = clipped[start : start + batch_size]
        out = np.asarray(use_embed_fn(batch), dtype=np.float32)
        chunks.append(out)
    emb = np.concatenate(chunks, axis=0)
    return np.stack([l2_normalize(row) for row in emb], axis=0)


review_vecs = encode_use(review_chunks["text"].tolist())
desc_app_ids = list(description_by_app.keys())
desc_vecs_arr = encode_use([description_by_app[a] for a in desc_app_ids])
description_vecs = {a: v for a, v in zip(desc_app_ids, desc_vecs_arr)}
print(f"review_vecs: {review_vecs.shape}, description_vecs: {len(description_vecs)}")

query_vecs = encode_use(examples_df["query_text"].tolist())
print(f"query_vecs: {query_vecs.shape}")

review_vecs: (15695, 512), description_vecs: 315
query_vecs: (12500, 512)


## Pool + Blend + Score the Full Grid

In [4]:
def pool_review_variant(review_df: pd.DataFrame, review_vecs: np.ndarray, *, polarity: str, weighting: str) -> dict[int, np.ndarray]:
    """Mirrors Stage 2's _pool_review_variant."""
    sub = review_df[review_df["recommended"] == 1] if polarity == "recommended_only" else review_df
    out: dict[int, np.ndarray] = {}
    for app_id, g in sub.groupby("app_id"):
        idx = g.index.to_numpy()
        vecs = review_vecs[idx]
        if weighting == "log_weighted":
            w = np.log1p(g["votes_helpful"].to_numpy(dtype=np.float64))
            if w.sum() <= 0:
                w = np.ones(len(g), dtype=np.float64)
        else:
            w = np.ones(len(g), dtype=np.float64)
        w = w / w.sum()
        out[int(app_id)] = l2_normalize((vecs * w[:, None]).sum(axis=0))
    return out


def blend_catalog(pooled_by_app: dict[int, np.ndarray], description_vecs: dict[int, np.ndarray], blend_weight: float) -> tuple[np.ndarray, np.ndarray]:
    """Mirrors Stage 2's _blend_with_description + the game_profiles write loop."""
    app_ids_out, vecs_out = [], []
    for app_id, desc_vec in description_vecs.items():
        pooled = pooled_by_app.get(app_id)
        if pooled is None:
            continue
        blended = l2_normalize((1.0 - blend_weight) * pooled + blend_weight * desc_vec)
        app_ids_out.append(app_id)
        vecs_out.append(blended)
    return np.asarray(app_ids_out, dtype=np.int64), np.stack(vecs_out, axis=0)


def score_examples(app_ids: np.ndarray, catalog_matrix: np.ndarray, query_vecs: np.ndarray) -> pd.DataFrame:
    app_to_row = {int(a): i for i, a in enumerate(app_ids)}
    rows = []
    for i, ex in enumerate(examples_df.itertuples(index=False)):
        scores = catalog_matrix @ query_vecs[i]
        self_row = app_to_row.get(int(ex.query_app_id))
        if self_row is not None:
            scores[self_row] = -np.inf
        ranked_rows = np.argsort(-scores)[:K_RETRIEVAL]
        rows.append(
            {
                "slice_name": ex.slice_name,
                "Hit@K": hit_rate_at_k(ranked_rows, ex.positives, K_RETRIEVAL, app_ids),
                "Recall@K": recall_at_k(ranked_rows, ex.positives, K_RETRIEVAL, app_ids),
            }
        )
    return pd.DataFrame(rows)


def summarize(df: pd.DataFrame) -> dict:
    overall_hit = df["Hit@K"].mean()
    by_slice = df.groupby("slice_name")[["Hit@K", "Recall@K"]].mean()
    return {
        "overall_hit": overall_hit,
        "slice_a_recall": by_slice.loc["slice_a_multi_target", "Recall@K"] if "slice_a_multi_target" in by_slice.index else float("nan"),
        "slice_b_hit": by_slice.loc["slice_b_single_target", "Hit@K"] if "slice_b_single_target" in by_slice.index else float("nan"),
    }


grid_rows = []
for polarity in POLARITY_ARMS:
    for weighting in WEIGHTING_ARMS:
        variant = f"{polarity}__{weighting}"
        pooled_by_app = pool_review_variant(review_chunks, review_vecs, polarity=polarity, weighting=weighting)
        for bw in BLEND_WEIGHTS:
            app_ids, catalog_matrix = blend_catalog(pooled_by_app, description_vecs, bw)
            metrics_df = score_examples(app_ids, catalog_matrix, query_vecs)
            s = summarize(metrics_df)
            grid_rows.append(
                {
                    "pooling_variant": variant,
                    "blend_weight": bw,
                    "n_games": len(app_ids),
                    "Hit@K (overall)": s["overall_hit"],
                    "Recall@K (Slice A)": s["slice_a_recall"],
                    "Hit@K (Slice B)": s["slice_b_hit"],
                }
            )
        print(f"scored variant={variant}")

grid_df = pd.DataFrame(grid_rows)
print(f"grid rows: {len(grid_df)}")

scored variant=any_polarity__flat
scored variant=any_polarity__log_weighted
scored variant=recommended_only__flat
scored variant=recommended_only__log_weighted
grid rows: 24


## Sanity Check: Reproduce `rag_v1`'s Real Baseline

In [5]:
sanity = grid_df[(grid_df["pooling_variant"] == "any_polarity__flat") & (grid_df["blend_weight"] == 0.1)].iloc[0]
print("This notebook's any_polarity__flat / blend_weight=0.1 cell:")
print(f"  Hit@K (overall)={sanity['Hit@K (overall)']:.3f}  Recall@K (Slice A)={sanity['Recall@K (Slice A)']:.3f}  Hit@K (Slice B)={sanity['Hit@K (Slice B)']:.3f}")
print("rag_v1's real eval_retrieval_overall.csv / eval_retrieval_by_slice.csv (rag_chunk_v1_query_plus_desc):")
print("  Hit@K (overall)=0.475  Recall@K (Slice A)=0.453  Hit@K (Slice B)=0.456")

This notebook's any_polarity__flat / blend_weight=0.1 cell:
  Hit@K (overall)=0.475  Recall@K (Slice A)=0.453  Hit@K (Slice B)=0.456
rag_v1's real eval_retrieval_overall.csv / eval_retrieval_by_slice.csv (rag_chunk_v1_query_plus_desc):
  Hit@K (overall)=0.475  Recall@K (Slice A)=0.453  Hit@K (Slice B)=0.456


## Comparison Grid vs. `two_tower_v1`

In [6]:
overall_csv = pd.read_csv(RUN_DIR / "eval_retrieval_overall.csv")
by_slice_csv = pd.read_csv(RUN_DIR / "eval_retrieval_by_slice.csv")
tt_overall_hit = overall_csv.loc[overall_csv["method"] == "two_tower_v1", "Hit@K"].iloc[0]
tt_slice_a_recall = by_slice_csv.loc[
    (by_slice_csv["method"] == "two_tower_v1") & (by_slice_csv["slice_name"] == "slice_a_multi_target"), "Recall@K"
].iloc[0]
tt_slice_b_hit = by_slice_csv.loc[
    (by_slice_csv["method"] == "two_tower_v1") & (by_slice_csv["slice_name"] == "slice_b_single_target"), "Hit@K"
].iloc[0]
print(f"two_tower_v1 bar: Hit@K (overall)={tt_overall_hit:.3f}  Recall@K (Slice A)={tt_slice_a_recall:.3f}  Hit@K (Slice B)={tt_slice_b_hit:.3f}")

grid_df["beats_two_tower_all_3"] = (
    (grid_df["Hit@K (overall)"] > tt_overall_hit)
    & (grid_df["Recall@K (Slice A)"] > tt_slice_a_recall)
    & (grid_df["Hit@K (Slice B)"] > tt_slice_b_hit)
)
display(grid_df.sort_values("Recall@K (Slice A)", ascending=False).round(3))

two_tower_v1 bar: Hit@K (overall)=0.512  Recall@K (Slice A)=0.460  Hit@K (Slice B)=0.496


                   pooling_variant  blend_weight  n_games  Hit@K (overall)  Recall@K (Slice A)  Hit@K (Slice B)  beats_two_tower_all_3
9       any_polarity__log_weighted          0.20      315            0.474               0.457            0.455                  False
4               any_polarity__flat          0.30      315            0.476               0.457            0.458                  False
3               any_polarity__flat          0.20      315            0.476               0.456            0.458                  False
2               any_polarity__flat          0.10      315            0.475               0.453            0.456                  False
10      any_polarity__log_weighted          0.30      315            0.474               0.453            0.456                  False
1               any_polarity__flat          0.05      315            0.474               0.451            0.455                  False
5               any_polarity__flat          0.50      3

## Key Findings

| pooling_variant | blend_weight | Hit@K (overall) | Recall@K (Slice A) | Hit@K (Slice B) |
|---|---|---|---|---|
| `two_tower_v1` (bar) | -- | 0.512 | 0.460 | 0.496 |
| **`any_polarity__flat` (best)** | **0.2-0.3** | **0.476** | **0.456-0.457** | **0.458** |
| `any_polarity__flat` (untuned, `rag_v1`) | 0.1 | 0.475 | 0.453 | 0.456 |
| `any_polarity__log_weighted` (best) | 0.3 | 0.474 | 0.453 | 0.456 |
| `recommended_only__flat` (best) | 0.3 | 0.471 | 0.448 | 0.454 |
| `recommended_only__log_weighted` (best) | 0.3 | 0.471 | 0.445 | 0.454 |

- Zero of 24 cells beat `two_tower_v1` on all three cuts.
- Blend weight: real but tiny lever, shallow optimum at 0.2-0.3 (+0.001-0.004 vs. default).
  Extremes (0.0, 0.5) hurt more.
- `log_weighted`/`recommended_only` underperformed plain `any_polarity__flat` across the board --
  opposite of predicted.
- Slice B Hit@K gap (0.458 vs 0.496) barely moves across any cell -- neither lever touches it.

## Recommendation / Next Steps

- Validates promoting `bge-small-en-v1.5` over tuning USE further -- USE-only tuning tops out
  ~0.04 short regardless of lever; the embedder swap alone closed the gap.
- `recs_024`'s confound is still open -- doesn't prove RAG beats two-tower architecturally, since
  `two_tower_v1` hasn't been retrained on bge-small. Real, expensive open question.
- Follow-up (not done): re-run `recs_024`'s bge-small comparison against this notebook's winning
  USE config instead of the untuned default -- expected to barely change the conclusion given how
  flat this grid was, but unverified.
- Vector-blended query construction (4th lever) remains untried -- lower priority now.